# Task 1: Binary sentiment classification using IMDB data.

Task 1 focuses on binary sentiment classification, aiming to predict whether a movie review is positive or negative. The IMDB dataset is well-suited for this task because it is conveniently preprocessed in tf.keras.datasets.imdb, allowing for immediate use. Since the data is composed of sequential word inputs, it is naturally compatible with sequence models such as RNNs, LSTMs, and 1D CNNs. Moreover, the dataset is small enough to be trained efficiently without the need for heavy computational resources.

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1) Load IMDB dataset
vocab_size = 10000  # keep top 10k words
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# 2) Get word index (mapping int -> word)
word_index = imdb.get_word_index()
index_to_word = {idx + 3: word for word, idx in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

# 3) Utility: decode review back to text
def decode_review(sequence):
    return " ".join([index_to_word.get(i, "?") for i in sequence])

# 4) Visualize a few samples
for i in range(5):
    text = decode_review(x_train[i])
    label = "positive" if y_train[i] == 1 else "negative"
    num_words = len(x_train[i])
    print(f"Sample {i+1}")
    print(f"Label: {label}")
    print(f"Text: {text[:300]}...")  # truncate for readability
    print(f"Num words: {num_words}")
    print("-" * 80)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Sample 1
Label: positive
Text: <START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the...
Num words: 218
--------------------------------------------------------------------------------
Sample 2
Label: negative
Text: <START> big hair big boobs bad music and a giant safety pin these are the words to best describe this terrible movie i love cheesy horror movies and i've seen hundreds but this had got to be on of the worst ever made the plot is paper thin and ridiculous the acting is an abomination the script is co...
Num words: 189
--------------------------------------------------------------------------------
Sample 3
Label: negative
Text: <START>

# Implementation using 1D CNN
**We use functional method here because:**
* We need non-linear architectures:
    * Multiple inputs (e.g., text + image)
    * Multiple outputs (e.g., classification + regression)
     * Skip connections (like ResNet)
    * Shared layers or branches (e.g., Siamese networks)
    * Arbitrary directed graph
* We want more explicit control over how tensors flow between layers.
* Flexible and explicit readability
* helpful for ResNet, Inception, multimodal models
* Think of it like drawing a computation graph — you can connect, merge, or branch paths freely.

Reason behind 256 max embedding size:
* most sentiment signals are contained within the first few hundred tokens (e.g., “This movie was fantastic...” or “The acting was awful…”).
    * So 256 tokens usually capture enough signal for classification accuracy >85–90%
* we need the fixed input length for RNNs, LSTMs, transformers. Padding to higher sizes could waste memory and slow training. If more required, can use hierarchical/attention models that can handle longer context
    * e.g. nuanced tasks like summarizatoin or topic analysis

In [3]:
# import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)

# 1) Data
# Define parameters
# Note: Underscore in numbers 1_000_000 etc is just Python's numeric literal separator for readability
vocab_size = 20_000 # Number of most frequent words to keep from the IMDB vocabulary - ignore rare words
max_len    = 256          # sequence length -- Each movie review will be truncated/padded to 256 tokens
embed_dim  = 16           # feature size per timestep (kept same as LSTM/RNN demos) --  Each token/word vector will be represented as a 16-dimensional vector ==> embedding layer output

# Load IMDB dataset (pre-tokenized to integers)
# Each review is now a list of word indices (integers representing tokens)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)
# Pad (or truncate) each sequence to 'max_len' tokens so all have the same length
# Short reviews are padded with zeros at the beginning ==> ensures every review has identical length (max_len = 256), which is required for batching
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

# 2) Model: 2-layer CNN1D (Conv -> Pool) x2, then Flatten -> Dense(1)
inputs = Input(shape=(max_len,), name="inputs")
x = Embedding(vocab_size, embed_dim, name="embed")(inputs) #Embedding Layer (20,000 × 16) = 320,000 parameters
x = Conv1D(32, 5, padding="same", activation="relu")(x) #Applies 32 filters of width 5 across the sequence, detecting local patterns (like "not bad" or "very good")
#Parameters: 32 × (5 × 16 + 1) = 2,592
x = MaxPooling1D(2)(x) #Downsamples by 2x, keeping strongest features, reduces 256 → 128

x = Conv1D(64, 5, padding="same", activation="relu")(x) #Applies 64 filters to detect higher-level patterns
#Parameters: 64 × (5 × 32 + 1) = 10,304
x = MaxPooling1D(2)(x) #Reduces 128 → 64

x = Flatten(name="flatten")(x) #Converts 64 × 64 = 4,096 values to 1D
x = Dense(64, activation="relu")(x) #Combines features for classification
outputs = Dense(1, activation="sigmoid")(x) #Binary sentiment output
# Parameters: 4,096 × 64 + 64 + 64 × 1 + 1 = 262,209

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# 3) Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,            # you can set to 10 if you prefer
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

# 4) Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Embedding)               │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 256, 32)        │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 128, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 595,169 (2.27 MB)

 Trainable params: 595,169 (2.27 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
157/157 - 5s - 29ms/step - accuracy: 0.7229 - loss: 0.4950 - val_accuracy: 0.8826 - val_loss: 0.2885
Epoch 2/5
157/157 - 3s - 22ms/step - accuracy: 0.9158 - loss: 0.2092 - val_accuracy: 0.8792 - val_loss: 0.3136
Epoch 3/5
157/157 - 3s - 22ms/step - accuracy: 0.9488 - loss: 0.1346 - val_accuracy: 0.8778 - val_loss: 0.3397
Test accuracy: 0.8766


In [4]:
# trying with more epochs
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=10,            # you can set to 10 if you prefer
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

# 4) Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/10
157/157 - 4s - 23ms/step - accuracy: 0.9140 - loss: 0.2112 - val_accuracy: 0.8804 - val_loss: 0.3199
Epoch 2/10
157/157 - 3s - 22ms/step - accuracy: 0.9471 - loss: 0.1418 - val_accuracy: 0.8912 - val_loss: 0.3059
Epoch 3/10
157/157 - 3s - 22ms/step - accuracy: 0.9595 - loss: 0.1030 - val_accuracy: 0.8234 - val_loss: 0.7566
Epoch 4/10
157/157 - 3s - 22ms/step - accuracy: 0.9744 - loss: 0.0710 - val_accuracy: 0.8616 - val_loss: 0.6134
Test accuracy: 0.8780


# Exercise 1a (LSTM):
Build an LSTM model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.

**Note** \
Our CNN’s ~600k parameters likely come from its dense layers (64 neurons + final layer).\
An LSTM’s parameter count grows quadratically with its hidden dimension (because it has 4 internal gates).\
Formula:\
For an LSTM with `units = h` and input size `d`, parameters =
`4 * [(d + h) * h + h] = 4 * h * (d + h + 1)`

In [15]:
from tensorflow.keras.layers import LSTM
# Define model architecture using the Functional API
inputslstm = Input(shape=(max_len,), name="inputs")                 # Input: sequence of 256 integers
xlstm = Embedding(vocab_size, embed_dim)(inputslstm)# Word embedding layer (20,000 × 16) = 320,000 parameters
xlstm = LSTM(256, return_sequences=False)(xlstm) # LSTM with 320 hidden units (~600K params)
xlstm = Dense(64, activation='relu')(xlstm) # Fully connected hidden layer
outputslstm = Dense(1, activation='sigmoid')(xlstm) # Binary output (positive/negative review)


modellstm = Model(inputs=inputslstm, outputs=outputslstm, name="LSTM_IMDB")
# Compile model with binary crossentropy (since it's a binary classification)
modellstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modellstm.summary() # View model summary (parameter counts, layer shapes)


Model: "LSTM_IMDB"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_9 (Embedding)         │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 256)            │       279,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 616,065 (2.35 MB)

 Trainable params: 616,065 (2.35 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
# Stop training if validation accuracy stops improving
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",   # metric to monitor
    patience=5,               # wait 5 epochs without improvement
    min_delta=0.001,          # ignore small fluctuations < 0.1%
    restore_best_weights=True,# revert to best weights when stopping
    verbose=1
)

callbacks_list = [early_stop]

history = modellstm.fit(
    x_train,
    y_train,
    validation_split=0.2,     # 20% of training data used for validation
    epochs=20,                # upper limit (EarlyStopping will likely stop earlier)
    batch_size=128,           # number of samples per gradient update
    callbacks=callbacks_list, # apply early stopping
    verbose=2                 # clean output (one line per epoch)
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

Epoch 1/20
157/157 - 69s - 442ms/step - accuracy: 0.5622 - loss: 0.6753 - val_accuracy: 0.8090 - val_loss: 0.4250
Epoch 2/20
157/157 - 68s - 430ms/step - accuracy: 0.8409 - loss: 0.3692 - val_accuracy: 0.8730 - val_loss: 0.3077
Epoch 3/20
157/157 - 68s - 432ms/step - accuracy: 0.9119 - loss: 0.2247 - val_accuracy: 0.8602 - val_loss: 0.4312
Epoch 4/20
157/157 - 68s - 430ms/step - accuracy: 0.9319 - loss: 0.1861 - val_accuracy: 0.8652 - val_loss: 0.3505
Epoch 5/20
157/157 - 68s - 430ms/step - accuracy: 0.9517 - loss: 0.1377 - val_accuracy: 0.8574 - val_loss: 0.4425
Epoch 6/20
157/157 - 68s - 430ms/step - accuracy: 0.9565 - loss: 0.1261 - val_accuracy: 0.8650 - val_loss: 0.4724
Epoch 7/20
157/157 - 68s - 435ms/step - accuracy: 0.9635 - loss: 0.1015 - val_accuracy: 0.8346 - val_loss: 0.5186
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 2.
782/782 - 24s - 30ms/step - accuracy: 0.5148 - loss: 0.6931
Test accuracy: 0.5148, Test loss: 0.6931


#### More LSTM variations

In [27]:
from tensorflow.keras.layers import Bidirectional

inputs = Input(shape=(256,), name="inputs")
x = Embedding(20_000, 16, name="embed")(inputs)

# Bidirectional LSTM with 64 units = 128 total features
# x = Bidirectional(LSTM(64, dropout=0.2))(x)

# Option 1: Larger Biderection LSTM
x = Bidirectional(LSTM(178, dropout=0.2))(x)  # ~600k total

# Option 2: Stacked LSTMs
# x = LSTM(64, return_sequences=True, dropout=0.2)(x)
# x = LSTM(64, dropout=0.2)(x)  # ~500k total

# Option 3: Larger embedding
# x = Embedding(20_000, 32, name="embed")(inputs)  # 640k in embedding alone
# x = LSTM(64, dropout=0.2)(x)

x = Dense(64, activation="relu")(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Embedding)               │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_8 (Bidirectional) │ (None, 356)            │       277,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 64)             │        22,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 620,593 (2.37 MB)

 Trainable params: 620,593 (2.37 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,     # 20% of training data used for validation
    epochs=20,                # upper limit (EarlyStopping will likely stop earlier)
    batch_size=128,           # number of samples per gradient update
    callbacks=callbacks_list, # apply early stopping
    verbose=2                 # clean output (one line per epoch)
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

Epoch 1/20
157/157 - 59s - 378ms/step - accuracy: 0.6377 - loss: 0.6378 - val_accuracy: 0.7272 - val_loss: 0.5316
Epoch 2/20
157/157 - 57s - 362ms/step - accuracy: 0.8389 - loss: 0.3625 - val_accuracy: 0.8430 - val_loss: 0.3549
Epoch 3/20
157/157 - 57s - 362ms/step - accuracy: 0.9200 - loss: 0.2093 - val_accuracy: 0.8578 - val_loss: 0.3276
Epoch 4/20
157/157 - 57s - 361ms/step - accuracy: 0.9449 - loss: 0.1490 - val_accuracy: 0.8466 - val_loss: 0.3801
Epoch 5/20
157/157 - 57s - 360ms/step - accuracy: 0.9597 - loss: 0.1157 - val_accuracy: 0.8376 - val_loss: 0.5052
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 1.
782/782 - 23s - 29ms/step - accuracy: 0.7363 - loss: 0.5273
Test accuracy: 0.7363, Test loss: 0.5273


# Exercise 1b (RNN):
Build an RNN model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.

**NOTE**\
To match our earlier CNN/LSTM (~600K parameters), we can scale the number of RNN units.\
A SimpleRNN has fewer parameters than LSTM because it lacks the four gating mechanisms.\
For an RNN with input dimension `d` and hidden size `h`:\
`params = (d + h + 1) * h`\
If your embedding output is embed_dim = 16 and you want ≈600K parameters:\
Solve `(16 + h + 1) * h ≈ 600,000` → `h ≈ 750`.\

In [30]:
from tensorflow.keras.layers import  SimpleRNN

# Model architecture
inputsrnn = Input(shape=(max_len,), name="inputs")
xrnn = Embedding(vocab_size, embed_dim, name="embedding")(inputsrnn)
xrnn = SimpleRNN(750, name="rnn")(xrnn)           # ≈600K parameters
xrnn = Dense(64, activation="relu", name="dense")(xrnn)
outputsrnn = Dense(1, activation="sigmoid", name="output")(xrnn)

modelrnn = Model(inputsrnn, outputsrnn, name="RNN_IMDB")
modelrnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelrnn.summary()


Model: "RNN_IMDB"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn (SimpleRNN)                 │ (None, 750)            │       575,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        48,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 943,379 (3.60 MB)

 Trainable params: 943,379 (3.60 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
history = modelrnn.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    callbacks=[early_stop],
    verbose=2
)

test_loss, test_acc = modelrnn.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

Epoch 1/20
157/157 - 90s - 575ms/step - accuracy: 0.5003 - loss: 0.9095 - val_accuracy: 0.5062 - val_loss: 2.1348
Epoch 2/20
157/157 - 89s - 569ms/step - accuracy: 0.5008 - loss: 0.7157 - val_accuracy: 0.4938 - val_loss: 0.6970
Epoch 3/20
157/157 - 89s - 567ms/step - accuracy: 0.5060 - loss: 0.7016 - val_accuracy: 0.4938 - val_loss: 0.6971
Epoch 4/20
157/157 - 89s - 567ms/step - accuracy: 0.5048 - loss: 0.7014 - val_accuracy: 0.4938 - val_loss: 0.6958
Epoch 5/20
157/157 - 89s - 569ms/step - accuracy: 0.5221 - loss: 0.6952 - val_accuracy: 0.5296 - val_loss: 0.6847
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 1.
782/782 - 53s - 68ms/step - accuracy: 0.5000 - loss: 2.1615
Test accuracy: 0.5000, Test loss: 2.1615


---


# Task 2: Reuters Newswire Topics (multi-class classification with 46 labels).

Task 2 involves predicting the topic category of a short newswire article among 46 possible classes. The Reuters dataset is conveniently available in tf.keras.datasets.reuters, so you can load and preprocess it immediately. Its sequences are typically shorter than those in IMDB, which makes training faster and well suited to classroom demos or quick iterations. Because it is a multi-class problem with compact inputs, Reuters is an excellent benchmark for comparing different model architectures—such as 1D CNNs, RNNs/LSTMs, and Transformers—on the same text-classification task.

In [9]:
import tensorflow as tf
from tensorflow.keras.datasets import reuters

# 1) Load Reuters dataset
vocab_size = 10000   # keep top 10k words
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=vocab_size)

# 2) Get word index mapping (int -> word)
word_index = reuters.get_word_index()
index_to_word = {idx + 3: word for word, idx in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

# 3) Reuters topic labels (from Reuters-21578 dataset, 46 classes)
reuters_topics = {
    0: "cocoa", 1: "grain", 2: "veg-oil", 3: "earn", 4: "acq",
    5: "wheat", 6: "corn", 7: "crude", 8: "money-fx", 9: "interest",
    10: "ship", 11: "trade", 12: "reserves", 13: "cotton", 14: "coffee",
    15: "sugar", 16: "gold", 17: "tin", 18: "strategic-metal", 19: "livestock",
    20: "retail", 21: "ipi", 22: "iron-steel", 23: "rubber", 24: "heat",
    25: "jobs", 26: "lei", 27: "money-supply", 28: "alum", 29: "oilseed",
    30: "gas", 31: "cpi", 32: "money-market", 33: "palm-oil", 34: "dmk-mark",
    35: "bop", 36: "gnp", 37: "silver", 38: "zinc", 39: "income",
    40: "lead", 41: "housing", 42: "copper", 43: "meal-feed", 44: "ipi-indicator",
    45: "strategic"
}

# 4) Utility: decode newswire back to text
def decode_newswire(sequence):
    return " ".join([index_to_word.get(i, "?") for i in sequence])

# 5) Visualize a few samples
for i in range(5):
    text = decode_newswire(x_train[i])
    label_id = y_train[i]
    label_name = reuters_topics.get(label_id, "unknown")
    num_words = len(x_train[i])
    print(f"Sample {i+1}")
    print(f"Label: {label_id} ({label_name})")
    print(f"Num words: {num_words}")
    print(f"Text: {text[:300]}...")  # truncate for readability
    print("*" * 80)


Sample 1
Label: 3 (earn)
Num words: 87
Text: <START> <UNK> <UNK> said as a result of its december acquisition of space co it expects earnings per share in 1987 of 1 15 to 1 30 dlrs per share up from 70 cts in 1986 the company said pretax net should rise to nine to 10 mln dlrs from six mln dlrs in 1986 and rental operation revenues to 19 to 22 ...
********************************************************************************
Sample 2
Label: 4 (acq)
Num words: 56
Text: <START> generale de banque sa lt <UNK> br and lt heller overseas corp of chicago have each taken 50 pct stakes in <UNK> company sa <UNK> factors generale de banque said in a statement it gave no financial details of the transaction sa <UNK> <UNK> turnover in 1986 was 17 5 billion belgian francs reut...
********************************************************************************
Sample 3
Label: 3 (earn)
Num words: 139
Text: <START> shr 3 28 dlrs vs 22 cts shr diluted 2 99 dlrs vs 22 cts net 46 0 mln vs 3 328 000 avg s

# Implementation using 1D CNN

In [10]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
import numpy as np

tf.random.set_seed(42)

# 1) Data (Reuters)
vocab_size = 20_000         # keep top words
max_len    = 256            # sequence length
embed_dim  = 16             # features per timestep

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.reuters.load_data(num_words=vocab_size)
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test  = tf.keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=max_len)

num_classes = int(max(y_train.max(), y_test.max()) + 1)
y_train = to_categorical(y_train, num_classes)
y_test  = to_categorical(y_test,  num_classes)

# 2) Model: 2-layer CNN1D (Conv -> Pool) x2, then Flatten -> Dense(num_classes)
inputs = Input(shape=(max_len,), name="inputs")
x = Embedding(vocab_size, embed_dim, name="embed")(inputs)
x = Conv1D(32, 5, padding="same", activation="relu")(x)
x = MaxPooling1D(2)(x)

x = Conv1D(64, 5, padding="same", activation="relu")(x)
x = MaxPooling1D(2)(x)

x = Flatten(name="flatten")(x)
x = Dense(64, activation="relu")(x)
outputs = Dense(num_classes, activation="softmax", name="out")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# 3) Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
]
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=5,                  # adjust as you like
    batch_size=128,
    callbacks=callbacks,
    verbose=2
)

# 4) Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Embedding)               │ (None, 256, 16)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 256, 32)        │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 128, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (Dense)                     │ (None, 46)             │         2,990 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 598,094 (2.28 MB)

 Trainable params: 598,094 (2.28 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
57/57 - 8s - 141ms/step - accuracy: 0.3878 - loss: 2.3695 - val_accuracy: 0.4736 - val_loss: 1.8998
Epoch 2/5
57/57 - 5s - 81ms/step - accuracy: 0.5368 - loss: 1.7442 - val_accuracy: 0.5748 - val_loss: 1.6257
Epoch 3/5
57/57 - 5s - 88ms/step - accuracy: 0.6145 - loss: 1.4705 - val_accuracy: 0.6027 - val_loss: 1.5577
Epoch 4/5
57/57 - 5s - 95ms/step - accuracy: 0.6605 - loss: 1.2648 - val_accuracy: 0.6144 - val_loss: 1.5733
Epoch 5/5
57/57 - 11s - 187ms/step - accuracy: 0.7118 - loss: 1.0877 - val_accuracy: 0.6138 - val_loss: 1.6306
Test accuracy: 0.6028  |  Test loss: 1.6451


# Exercise 2a (LSTM) (Optional):
Build an LSTM model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.


# Exercise 2b (RNN)  (Optional):
Build an RNN model with a comparable number of parameters to the given 1D CNN model. You may reuse some of the settings and parameters from the provided 1D CNN demo code.

---


# Task 3: Shakespeare / Tiny Shakespeare Character Dataset (character-level language modeling).
Task 3 focuses on next-character prediction, where the model learns to generate text one character at a time by predicting the next character in a sequence. The Tiny Shakespeare dataset is a classic benchmark for demonstrating RNNs and LSTMs because it is small (less than 1 MB), fun to work with. It provides a simple yet effective way to showcase how sequence models can capture language patterns and generate coherent text continuations.

# Visualization of text samples

In [11]:
import tensorflow as tf
import io

# 1) Download Tiny Shakespeare text
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
path = tf.keras.utils.get_file("tiny_shakespeare.txt", origin=url)
text = io.open(path, encoding="utf-8").read()

print("Total characters in corpus:", len(text))

# 2) Build char vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Unique characters (vocab size):", vocab_size)
print("Character set:", chars)

# 3) Map char <-> int
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

# 4) Encode the whole text
encoded = [stoi[c] for c in text]

# 5) Visualize a few samples
print("\n=== Sample visualization ===")
for i in range(3):
    snippet = text[i*200:(i+1)*200]   # take 200-char snippets
    encoded_snippet = encoded[i*200:(i+1)*200]
    print(f"\nSample {i+1}")
    print("Raw text:\n", snippet[:300].replace("\n", "\\n"))  # replace newline for clarity
    print("Encoded IDs:\n", encoded_snippet[:50], "...")      # show first 50 IDs
    print("*" * 80)


Total characters in corpus: 1115394
Unique characters (vocab size): 65
Character set: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

=== Sample visualization ===

Sample 1
Raw text:
 First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you
Encoded IDs:
 [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56] ...
********************************************************************************

Sample 2
Raw text:
  know Cai

# Exercise 3a (LSTM) (Optional):
Build an LSTM model for next-character prediction.

# Exercise 3b (RNN) (Optional):
Build an RNN model for next-character prediction.